# Session 2: GLM, RSA, and Functional Connectivity

In [sess-1a](sess-1a.ipynb) we learned how raw scanner data are converted to NIfTI and organised into a **BIDS** dataset, and how to load and inspect images with **nilearn**. In [sess-1b](sess-1b.ipynb) we used **MRIQC** to assess raw-data quality and **fMRIprep** to preprocess the BOLD time series.

We now move from data handling to **statistical analysis**. The lesson is organised around **three main themes**, all illustrated on the same dataset where possible:

  1. **First-level + group GLM**: fitting a linear model at every voxel to ask *where* and *how strongly* the brain responds to an experimental condition, in a single subject and across the group.

  2. **Functional Connectivity (FC)**: keeping the same dataset but discarding the events, asking instead *which regions co-fluctuate over time* and visualising the resulting brain network.

  3. **Representational Similarity Analysis (RSA)**: turning multivariate activity patterns into a *similarity space* and comparing it to theoretical models of how the brain might represent stimuli (with a richer dataset).

Each method uses its own canonical demo dataset, all of which are downloaded automatically the first time you run the notebook:

| Section | Topic | Dataset | Key library |
|---|---|---|---|
| 1 | First-level + group GLM | nilearn language localizer (10 subj, BIDS + preprocessed in MNI) | `nilearn.glm`, `pybids` |
| 2 | Functional connectivity | **same** language localizer (1 subj, MSDL atlas) | `nilearn.connectome` |
| 3 | RSA | Algonauts 2019 / Cichy et al. (15 subj, 92 stimuli) | `rsatoolbox` |

> **Before you start.** This notebook assumes you have completed sess-1a and sess-1b. If VS Code asks for a kernel, pick `base (Python 3.13.12)` (the same one you used in the previous sessions).

Icons used throughout, identical to sess-1a / sess-1b:

| Icon | Meaning |
|---|---|
| 🖥️ | **Switch to Neurodesktop**: you will need to open the Neurodesktop environment for this section |
| 🐍 | **Python**: this section contains code cells to run in the notebook |
| 📝 | **Practice**: exercises and tasks for you to complete |


In [ ]:
# `%pip` is the notebook magic for installing Python packages into the
# current kernel (same as in sess-1b). We need:
#   - nilearn:    fMRI loading, GLM, plotting, connectivity
#   - pybids:     query a BIDS dataset like a database (already used in sess-1a)
#   - rsatoolbox: representational similarity analysis
%pip install -q nilearn pybids rsatoolbox


In [ ]:
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# All per-run outputs and copies in this session land under results/sess-2/
# so you can wipe the folder and re-run from scratch at any time. The data
# itself lives on the shared course storage at /data/teaching/costantinoai/sess-2/
# and the %%bash cells in each section copy it from there into results/.
RESULTS_DIR = Path('results/sess-2')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Confirm versions, useful when sharing or debugging the notebook.
from importlib.metadata import version
for pkg in ('nilearn', 'pybids', 'rsatoolbox'):
    print(f'{pkg:<12s} {version(pkg)}')


## 1. GLM with pyBIDS and nilearn (first and second level)

### 1.1 What a GLM does

For every voxel in the brain we have a noisy time series, the BOLD signal. The General Linear Model asks a simple question: *can this time series be explained as a weighted sum of known regressors, plus noise?*

$$Y = X\beta + \varepsilon$$

<img src="assets/sess-2/videos/slide-design-matrix.png" width="900" alt="GLM as a matrix equation: Y = X beta + epsilon"/>

* $Y$, the BOLD signal at one voxel (one number per TR).
* $X$, the **design matrix**: each column is one regressor (an expected response for one experimental condition, plus nuisance signals like head motion).
* $\beta$, the unknown weights we want to estimate. **One $\beta$ per voxel per regressor**: a large $\beta$ for the *faces* regressor at a given voxel means that voxel responds more strongly during face trials than during the implicit baseline.
* $\varepsilon$, what the model could not explain (residuals).

A first-level GLM fits this equation **independently at every voxel**, producing one $\beta$-map (whole brain, each voxel a beta value) per regressor. These $\beta$-maps are **effect-size estimates**, not yet significance maps. We then form **contrasts** (e.g. $\beta_{\text{faces}} - \beta_{\text{houses}}$) to ask a hypothesis-testing question and turn the result into a statistical map.

> **The three building blocks.** Everything in the rest of Section 1 is one of these:
> 1. The **events**, i.e. when each condition happened (`events.tsv`).
> 2. The **confounds**, i.e. nuisance regressors we want to project out (head motion, framewise displacement, drift) from the fMRIPrep output.
> 3. The **HRF model**, i.e. the canonical haemodynamic response we convolve event onsets with to build the expected BOLD response.

### 1.2 🐍 The BIDS demo dataset

We use nilearn's [`fetch_language_localizer_demo_dataset`](https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_language_localizer_demo_dataset.html), a small demo dataset with **fMRIPrep derivatives already computed**. It has the same structure you would get by running fMRIPrep yourself on a task experiment (exactly like the ds000117 derivatives you explored in sess-1b).

The dataset is pre-staged on the shared course storage at `/data/teaching/costantinoai/sess-2/`. The cell below copies it into your `results/sess-2/` folder, the same pattern you used in sess-1b §1.1.


In [ ]:
%%bash
# Copy the language-localizer demo dataset from the shared course folder into
# your personal results/ folder. Same pattern as sess-1b: shared storage is
# read-only, results/ is yours to read and write freely.
mkdir -p results/sess-2
rm -rf  results/sess-2/fMRI-language-localizer-demo-dataset
cp  -r  /data/teaching/costantinoai/sess-2/fMRI-language-localizer-demo-dataset \
        results/sess-2/
chmod -R u+w results/sess-2/fMRI-language-localizer-demo-dataset


In [ ]:
# Point nilearn at the local copy. The fetch helper just resolves paths
# inside data_dir without re-downloading because the dataset is already there.
# docs: https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_language_localizer_demo_dataset.html
from nilearn.datasets import fetch_language_localizer_demo_dataset

data = fetch_language_localizer_demo_dataset(data_dir=str(RESULTS_DIR))
data_dir = Path(data.data_dir)
print('BIDS root:', data_dir)


In [ ]:
%%bash
# Show the BIDS layout with `tree`. Depth 3 surfaces:
#   - the dataset-level files (README, CHANGES, dataset_description.json, participants.tsv)
#   - the 10 subject folders (sub-01 ... sub-10)
#   - the parallel `derivatives/` folder that fMRIPrep wrote, with the same per-subject layout
# Same idea you saw in sess-1b §3: a BIDS dataset that holds its own derivatives.
tree results/sess-2/fMRI-language-localizer-demo-dataset


The key thing to notice is the split between the raw `func/` files and the `derivatives/` outputs that the GLM will actually consume (`*_desc-preproc_bold.nii.gz`, `*_desc-brain_mask.nii.gz`, `*_desc-confounds_*.tsv`). Same idea you saw in sess-1b §3.

Take a minute to explore the BIDS structure in the Explorer tab on the left. Go to `results/sess-2/fMRI-language-localizer-demo-dataset`

> ❓ **Questions:**
>
> 1. How many subjects are in this dataset?
>
> 2. How many runs per subject?
>
> 3. What type of derivatives do we have, and what are they?
>
> 4. Open one `events.tsv` file. What conditions are present in this task? How many TRs per run?

### 1.3 🐍 Querying the dataset with pyBIDS

Rather than constructing paths by hand, we use [`BIDSLayout`](https://bids-standard.github.io/pybids/generated/bids.layout.BIDSLayout.html) to query the dataset by **entities** (subject, task, run, desc, ...). This is exactly the `BIDSLayout(...).get(...)` pattern you used in sess-1a §3.4 and sess-1b §3, and the same code would work on a 200-subject dataset.


In [ ]:
from bids import BIDSLayout

# `derivatives=True` indexes the fMRIPrep folder alongside the raw data.
# `validate=False` skips strict BIDS validation (the demo lacks a few sidecars).
# docs: https://bids-standard.github.io/pybids/generated/bids.layout.BIDSLayout.html
layout = BIDSLayout(data_dir, derivatives=True, validate=False)

print('Subjects:', layout.get_subjects())
print('Tasks:   ', layout.get_tasks())
print('Runs:    ', layout.get_runs() or '(only one run per subject)')


In [ ]:
# Pick one subject and pull the three files we need for the GLM.
subject = '01'
task    = 'languagelocalizer'

# These are the actual nifti images
bold_file      = layout.get(subject=subject, task=task, scope='derivatives',
                            desc='preproc', extension='.nii.gz',
                            return_type='filename')[0]
# This is the event file                            
events_file    = layout.get(subject=subject, task=task, scope='raw',
                            suffix='events', extension='.tsv',
                            return_type='filename')[0]
confounds_file = layout.get(subject=subject, task=task, scope='derivatives',
                            desc='confounds', extension='.tsv',
                            return_type='filename')[0]

# Print the filenames
for label, p in [('bold     ', bold_file),
                 ('events   ', events_file),
                 ('confounds', confounds_file)]:
    print(f'{label}: {Path(p).name}')


Three queries, three files. Notice how each query reads almost like a sentence: *give me the preprocessed BOLD for subject 01, task languagelocalizer, in the derivatives scope.*

### 1.4 🐍 Plotting the events

Now we have our datasets and its derivatives nicely indexed in pyBIDS.

Before fitting any model, always **look at your events**. Mistimed onsets, wrong durations, or missing trials will silently corrupt every analysis downstream.

[`plot_event`](https://nilearn.github.io/stable/modules/generated/nilearn.plotting.plot_event.html) takes a list of event DataFrames (one per run) and stacks them as subplots, so the same call generalises to multi-run experiments.


In [ ]:
from nilearn.plotting import plot_event

# Read the events.tsv: it has one row per trial with onset / duration / trial_type.
events = pd.read_csv(events_file, sep='\t')
print(events.head())
print(f'\n{len(events)} trials  |  conditions: {sorted(events.trial_type.unique())}')

# plot_event accepts a list of DataFrames, one entry per run. Here we have a
# single run, but the loop pattern works unchanged for many.
# docs: https://nilearn.github.io/stable/modules/generated/nilearn.plotting.plot_event.html
fig = plot_event([events], figsize=(14, 2.5))
fig.suptitle(f'sub-{subject} task-{task}: events', y=1.05)
plt.show()


In the plot above, we see all trials presented in one run. The y axis indicates the presence of the condition, and stimuli extend on the x axis for how long they are presented. Notice how trials are balanced, have similar durations, and include some fixation/rest time between presentations.

> ❓ **Questions:**
>
> 1. Read the table printed above. How long is each stimulus presented? And how long is each fixation/rest period?
>
> 2. If you were to repeat this study, would you keep the timings similar, or would you make fixation times longer or shorter? Why?

### 1.5 🐍 fMRIPrep confounds: 6 motion parameters

Subjects move. Even tiny movements introduce signal changes that look a lot like real activations and will inflate false-positive rates if left in the model. fMRIPrep writes a confounds table per run; here we keep the rigid-body **6 head motion parameters** (3 translations + 3 rotations) and use them as nuisance regressors. 

This is the same minimal motion model you computed by hand in sess-1b §3.6, just bundled into the GLM rather than plotted on its own. By adding these confound/nuisance regressors to our model, we are "removing" from the data part of the effect related to those sources of noise.

> **Why these and not more?** A larger expansion (e.g. Friston-24, aCompCor, scrubbing) is appropriate for many studies, see [`load_confounds_strategy`](https://nilearn.github.io/stable/modules/generated/nilearn.interfaces.fmriprep.load_confounds_strategy.html) for nilearn's recommended presets. Here we stay minimal so you can see exactly what enters the design matrix.


In [ ]:
# Read the full confounds table.
all_confounds = pd.read_csv(confounds_file, sep='\t')
print(f'fMRIPrep wrote {all_confounds.shape[1]} confound columns. We keep the 6 motion terms:')

# This particular demo was generated with an older fMRIPrep that uses the
# names X / Y / Z / RotX / RotY / RotZ. Modern fMRIPrep (and sess-1b) uses
# trans_x / trans_y / trans_z / rot_x / rot_y / rot_z. Rename so the rest of
# the notebook is consistent with what you saw in sess-1b.
rename = {'X': 'trans_x', 'Y': 'trans_y', 'Z': 'trans_z',
          'RotX': 'rot_x', 'RotY': 'rot_y', 'RotZ': 'rot_z'}
motion_cols = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']

confounds = all_confounds.rename(columns=rename)[motion_cols].copy()
print(confounds.describe().round(3))

For this subject motion is well-behaved (sub-millimetre translations, sub-degree rotations). Compare the `std` and peak` of these traces to sub-03 run-08 in sess-1b §3.6, where the spikes were obvious by eye.

### 1.6 🐍 Building the design matrix

The design matrix $X$ has one row per TR and one column per regressor. We build it with [`make_first_level_design_matrix`](https://nilearn.github.io/stable/modules/generated/nilearn.glm.first_level.make_first_level_design_matrix.html), which:

1. Convolves each event type's onset/duration boxcar with a canonical **HRF** to get one expected-response regressor per condition.
2. Appends our **confounds** as additional columns (`add_regs=...`).
3. Adds an intercept column.


In [ ]:
import nibabel as nib
from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix
from scipy.stats import zscore

# Read TR and number of volumes from the BOLD image header so the design
# matrix lines up with the actual acquisition.
img = nib.load(bold_file)
t_r = float(img.header.get_zooms()[-1])
n_volumes = img.shape[-1]
frame_times = np.arange(n_volumes) * t_r
print(f'TR = {t_r} s  |  {n_volumes} volumes  |  run length = {n_volumes * t_r:.1f} s')

# Build the first-level design matrix. Each column is one regressor the GLM
# will estimate:
#   - condition columns model the expected HRF-shaped task response
#   - confound columns model motion-related variance
#   - the constant captures the mean level
design_matrix = make_first_level_design_matrix(
    frame_times,
    events=events,
    drift_model=None,
    hrf_model='spm',                  # canonical SPM HRF
    add_regs=zscore(confounds.values, axis=0),         # 6 motion terms, zscored
    add_reg_names=motion_cols,
)
print(f'design matrix: {design_matrix.shape[0]} TRs x {design_matrix.shape[1]} regressors')
print('columns:', list(design_matrix.columns))

# Visual check: the task columns should have HRF-shaped bumps.
fig, ax = plt.subplots(figsize=(8, 6))
plot_design_matrix(
    design_matrix,
    rescale=True,
    axes=ax)
ax.set_title('First-level design matrix'); plt.show()

> ❓ **Questions:**
>
> 1. What do the colors indicate in the `rot_*` and `trans_*` columns? What about in `language` and `string`?
>
> 2. The condition columns are **HRF-convolved** boxcars. Why convolve at all, instead of just using the raw on/off boxcar of when each block was on?
>
> 3. Imagine the language and string regressors were almost perfectly correlated (e.g. if the two conditions alternated very fast and the HRF blurred them together). What would happen to the beta estimates? (Hint: think about what regression does when two predictors carry the same information.)


Read this matrix left to right: each column is one regressor the model will estimate. The condition columns are the HRF-predicted response shapes; their coefficients are the $\beta$ values we inspect next. The motion columns soak up nuisance variance, and the intercept captures the overall mean.

### 1.7 🐍 Fitting the GLM and computing a contrast

Now we fit and ask a question. Which voxels are more active in the language condition? In other words: how much of the signal changes we measure in each voxel can be explained by the regressors in our design matrix?

The language localizer task contrasts **language** (`language`) against **string** stimuli (visual letter strings without linguistic content). The contrast `language - string` should peak in the classic left-lateralised language network.

Let's fit our GLM:

In [ ]:
from nilearn.glm.first_level import FirstLevelModel
from nilearn.glm import threshold_stats_img
from nilearn.plotting import plot_stat_map, view_img

# Fit a first-level GLM with 5 mm smoothing.
# - smoothing_fwhm=5 averages each voxel with its neighbours, which usually
#   improves SNR for group analyses on this kind of dataset.
# - mask_img=None (the default) lets nilearn estimate an implicit brain mask
#   from the BOLD itself.
# docs: https://nilearn.github.io/stable/modules/generated/nilearn.glm.first_level.FirstLevelModel.html
smoothed_model = FirstLevelModel(smoothing_fwhm=5)
smoothed_model.fit(bold_file, design_matrices=[design_matrix])

# Build a vector contrast from the design-matrix columns: +1 on language, -1 on string.
contrast_vec = np.zeros(design_matrix.shape[1])
contrast_vec[list(design_matrix.columns).index('language')] =  1
contrast_vec[list(design_matrix.columns).index('string')]   = -1

# z-map for the language-minus-string contrast.
smoothed_z = smoothed_model.compute_contrast(contrast_vec, output_type='z_score')

# We are interested in voxels that respond MORE to language than to strings,
# so we visualise the positive tail only. `threshold_stats_img` with
# `two_sided=False` thresholds at a principled alpha (here voxel-wise p<0.001,
# uncorrected) and zeroes out negatives in one call, no math_img needed.
# The underlying `smoothed_z` still carries both signs and is what we save and
# pass to MRIcron in §1.8; the figure below is one-sided.
smoothed_z_thr, z_thr = threshold_stats_img(
    smoothed_z, alpha=0.001, height_control='fpr', two_sided=False,
)

plot_stat_map(
    smoothed_z_thr,
    threshold=z_thr,
    display_mode='z',
    cut_coords=6,
    cmap='hot',
    symmetric_cbar=False,
    title=f'language > string (z+, p<0.001 unc), 5 mm smoothing, sub-{subject}',
)
plt.show()


The slices already hint at the story: warm-coloured clusters in **left inferior frontal gyrus** and along the **left superior / middle temporal gyrus**, with little on the right. That is the canonical left-lateralised language network. To explore the volume more freely, render the same z-map as an interactive viewer below.


In [ ]:
# One interactive viewer for the canonical (smoothed) result so you can
# scroll through the brain and pick a threshold by eye.
view_img(smoothed_z_thr, threshold=z_thr, cmap='hot', symmetric_cmap=False,
         title=f'language > string (z+), 5 mm smoothing, sub-{subject}')


### 1.8 🖥️ Practice: read the unthresholded betas in MRIcron

We have looked at the **contrast** $z$-map ($\beta_{\text{language}} - \beta_{\text{string}}$, thresholded). That figure tells us which voxels are significantly *more* active during the `language` condition, but it hides two things from you:

1. The contrast is the **difference of two underlying maps**. Each underlying map ($\beta_{\text{language}}$, $\beta_{\text{string}}$) is itself an answer to a question: *how strongly does each voxel respond to this single condition?*
2. The contrast was **thresholded**. Below the cutoff voxels look blank, but they are not zero, they are just smaller. Looking at the raw, unthresholded $\beta$s is the only way to see what the GLM actually estimated.

The cell below saves the two $\beta$ maps to disk so you can open them in MRIcron and *right-click voxels to read the actual numbers*. The right-click skill we use here is the same skill you used in [sess-1a §2.3](sess-1a.ipynb) to read atlas labels, except now the values are continuous response strengths.


In [ ]:
# Compute the two per-condition beta maps. output_type='effect_size' returns
# the beta in the GLM's natural units (close to percent signal change for this
# preprocessing pipeline). We do not threshold or smooth them further: the
# point of this exercise is to inspect raw values voxel by voxel.
language_beta = smoothed_model.compute_contrast('language', output_type='effect_size')
string_beta   = smoothed_model.compute_contrast('string',   output_type='effect_size')

# Save both to disk, alongside the contrast z-map saved above. We do NOT plot
# them inline: the figures we want to look at live in MRIcron, where you can
# right-click voxels.
language_beta_path = RESULTS_DIR / f'sub-{subject}_task-{task}_beta-language.nii.gz'
string_beta_path   = RESULTS_DIR / f'sub-{subject}_task-{task}_beta-string.nii.gz'
language_beta.to_filename(language_beta_path)
string_beta.to_filename(string_beta_path)

print('saved (open these in MRIcron):')
print(' ', language_beta_path.name)
print(' ', string_beta_path.name)


> 📝 **Exercise: right-click your way through the betas.**
>
> 1. Switch to **Neurodesktop** in your browser.
> 2. Open **MRIcron** (Applications -> Neurodesk -> Visualisation -> mricronGUI).
> 3. **File -> Open** and load an MNI152 template as the background (the one you copied in sess-1a is at `results/sess-1a/MNI152_T1_1mm.nii.gz`, or any MNI152 you have on disk).
> 4. **Overlay -> Add** and load **`sub-01_task-languagelocalizer_beta-language.nii.gz`** from `results/sess-2/`. In the overlay panel, **set the lower threshold to 0** so no voxels are hidden.
> 5. **Right-click on a voxel** anywhere in the brain. MRIcron prints the $\beta$ value for that voxel at the bottom of the window. *That number is what the GLM estimated as this voxel's response to the language condition.*
> ---
>
> **Connecting the dots: this is what RSA looks at.** In [§3](#3.-Representational-Similarity-Analysis-(RSA)) we will do the same right-click-and-read step in spirit, but in batch:
>
> * Pick an ROI (EVC or IT).
> * For each condition, **collect the $\beta$ value of every voxel inside that ROI into one row vector**. That row vector is what RSA calls a **multivariate pattern** or representation: a *signature* of the condition in that region. The number you read by right-click in MRIcron is *one entry* of one such row vector.
> * Stack one row per condition, get a (n_conditions × n_voxels_in_ROI) matrix of patterns.
> * Compute pairwise distances between rows, get an RDM, compare it to model RDMs.
>
> The substrate is identical: *the same $\beta$ values you are about to inspect*. We are not switching to a different kind of data; we are just looking at it through different glasses. **What we lose** going from a brain map to a pattern: voxel positions in 3D space (the spatial map collapses into a 1D vector). **What we gain**: a representation we can *compare* across regions and across hypotheses, which is what RSA is for.



### 1.9 🐍 From one subject to a group: second-level GLM

So far we fit one subject. A first-level $\beta$ map answers *where does this person's brain respond more to language than to strings?* But individual brains are noisy, and any one subject's map mixes real signal with that subject's idiosyncrasies. To make a claim about **language processing in general**, we need a **second-level** (group) analysis.

The recipe has two steps:

1. **First level, repeated per subject.** For every subject, fit the same GLM and compute the same contrast, this time saving the **effect-size** map (the $\beta$ image), not the z-map. We want the size of the effect, in the same units, so we can compare it across people.
2. **Second level.** Stack all subjects' effect-size maps and fit a one-sample t-test at every voxel: *is the mean effect across subjects different from zero?* This is itself a tiny GLM with a single column of ones (the group intercept). The output is a group-level z-map.

In [ ]:
from nilearn.glm.first_level import FirstLevelModel

# Loop over all subjects in the dataset and fit the same first-level GLM
# for each. We collect the effect-size map (the language - string beta)
# for every subject, which becomes the input to the second-level model.
subjects = layout.get_subjects()
print(f'fitting first level for {len(subjects)} subjects: {subjects}')

effect_maps = []
for sub in subjects:
    # Same three queries as in 1.3, just parameterised by subject.
    bold = layout.get(subject=sub, task=task, scope='derivatives',
                      desc='preproc', extension='.nii.gz', return_type='filename')[0]
    evs  = pd.read_csv(layout.get(subject=sub, task=task, scope='raw',
                       suffix='events', extension='.tsv', return_type='filename')[0], sep='\t')
    cnf  = pd.read_csv(layout.get(subject=sub, task=task, scope='derivatives',
                       desc='confounds', extension='.tsv', return_type='filename')[0], sep='\t')
    cnf  = cnf.rename(columns=rename)[motion_cols]

    # Same TR and design matrix recipe as in 1.6.
    img = nib.load(bold) # load the image
    n_vol = img.shape[-1] # get the TRs
    ft = np.arange(n_vol) * float(img.header.get_zooms()[-1])
    dm = make_first_level_design_matrix(ft, events=evs, hrf_model='spm',
                                        add_regs=cnf.values, add_reg_names=motion_cols)

    # Fit and grab the effect-size map for the language - string contrast.
    model = FirstLevelModel(smoothing_fwhm=5).fit(bold, design_matrices=[dm])
    cvec = np.zeros(dm.shape[1])
    cvec[list(dm.columns).index('language')] =  1
    cvec[list(dm.columns).index('string')]   = -1
    effect_maps.append(model.compute_contrast(cvec, output_type='effect_size'))
    print(f'  sub-{sub} done')

print(f'collected {len(effect_maps)} effect-size maps')

`effect_maps` now holds one **effect-size** NIfTI per subject (one per `subject` returned by `layout.get_subjects()`). They all live on the same MNI grid (the dataset normalised them in pre-processing), so a voxelwise mean across subjects is well-defined. That is exactly what the second-level GLM does next, except it returns a *t* / *z* statistic instead of a plain mean so we know whether the average effect is reliably non-zero.


In [ ]:
from nilearn.glm.second_level import SecondLevelModel

# The second-level design is a single column of ones: a group intercept.
# Fitting it asks, voxel by voxel, is the mean effect across subjects
# different from zero?
group_design = pd.DataFrame({'intercept': np.ones(len(effect_maps))})

second_level = SecondLevelModel(smoothing_fwhm=5).fit(
    effect_maps,
    design_matrix=group_design
    )

group_z = second_level.compute_contrast(
    second_level_contrast='intercept',
    output_type='z_score'
    )

# One-sided positive thresholding via threshold_stats_img: zeros out the
# negative tail and sub-threshold voxels in one call,
# and gives a principled cutoff (voxel-wise p<0.001, uncorrected).
group_z_thr, group_thr = threshold_stats_img(
    group_z, alpha=0.001, height_control='fpr', two_sided=False
)

# Plot the group map alongside the single-subject map from 1.7 so you can
# see how averaging across subjects sharpens the language network.
plot_stat_map(
    group_z_thr,
    threshold=group_thr,
    display_mode='z',
    cut_coords=6,
    cmap='hot',
    symmetric_cbar=False,
    title=f'group (n={len(effect_maps)}): language > string (z+, p<0.001 unc)',
    )

plt.show()

view_img(
    group_z_thr,
    threshold=group_thr,
    cmap='hot',
    symmetric_cmap=False,
    title=f'group (n={len(effect_maps)}): language > string (z+)',
    )


A picture is helpful but not specific: *"there's a blob around there"* is hard to communicate in a paper. The standard companion is a **cluster table** with anatomical labels. We use [`get_clusters_table`](https://nilearn.github.io/stable/modules/generated/nilearn.reporting.get_clusters_table.html) to extract suprathreshold blobs from the group z-map, then look up each peak's MNI coordinate in the Harvard-Oxford atlas (the same atlas you used in sess-1a §2.5) to put a region name on each row.


In [ ]:
from nilearn.datasets import fetch_atlas_harvard_oxford
from nilearn.reporting import get_clusters_table
from nilearn.image import resample_to_img

# Same Harvard-Oxford atlas you met in sess-1a (cortical, max-probability,
# thresholded at 25%, 2 mm). It is a label image: each voxel holds an
# integer index into atlas.labels.
# docs: https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_atlas_harvard_oxford.html
atlas = fetch_atlas_harvard_oxford('cortl-maxprob-thr25-2mm', data_dir=str(RESULTS_DIR))

# Cluster table: one row per cluster, with sub-peaks listed beneath.
# X / Y / Z are MNI coordinates in millimetres.
# docs: https://nilearn.github.io/stable/modules/generated/nilearn.reporting.get_clusters_table.html
clusters = get_clusters_table(group_z, stat_threshold=2, cluster_threshold=20)

# Resample the atlas onto the z-map grid so MNI coordinates map to the
# same voxel indices in both images, then look up the atlas label at each
# peak coordinate.
atlas_on_z = resample_to_img(atlas.maps, group_z, interpolation='nearest',
                             force_resample=True, copy_header=True)

atlas_data = atlas_on_z.get_fdata().astype(int)
inv_affine = np.linalg.inv(atlas_on_z.affine)

def label_at(x, y, z):
    i, j, k = np.round(inv_affine @ [x, y, z, 1])[:3].astype(int)
    return atlas.labels[atlas_data[i, j, k]]

clusters['region'] = [label_at(r['X'], r['Y'], r['Z']) for _, r in clusters.iterrows()]
print(clusters[['Cluster ID', 'X', 'Y', 'Z', 'Peak Stat', 'region']].to_string(index=False))

The group map should show the same left-lateralised language network as the single-subject map, but cleaner and more focal. That is what a group analysis buys you: subject-specific noise averages out, the shared signal survives. The peak table puts a name on each blob: expect labels in the **Frontal Pole / Inferior Frontal Gyrus** family on the left, and **Middle / Superior Temporal Gyrus** on the left.

> ❓ **Questions:**
>
> 1. We passed effect-size maps to `SecondLevelModel`, not z-maps. What would go wrong if we passed z-maps instead?
>
> 2. The second-level design matrix has a single column of ones. Sketch the design matrix you would use to compare two groups (e.g. patients vs controls) at the second level.
>
> 3. With $n=10$ subjects and a one-sample t-test, the second-level $t$ has only 9 degrees of freedom. How does that affect what counts as a significant voxel, compared to the first-level fit which had hundreds of TRs?

## 2. Functional Connectivity (FC) on the language-localizer data

### 2.1 The FC idea

The GLM in [§1](#1.-GLM-with-pyBIDS-and-nilearn-(first-and-second-level)) asked *where* the brain responds to language. **FC asks a complementary question**: which regions co-fluctuate over time, regardless of any task model?

> 💡 **Intuition.** Forget the events for a moment. Each voxel's BOLD signal rises and falls across the 5-minute scan. If two regions rise and fall at the same moments, they are *functionally coupled*; if they wiggle independently, they are not. FC turns that pairwise observation into a graph of the brain.

There are two standard ways to draw that graph:

* **Seed-to-voxel.** Pick one region, ask which voxels share its time course. The output is one whole-brain correlation map, the seed's *connectivity profile*. Good when you have a hypothesis about a specific region (e.g. "what does left IFG talk to?").
* **Whole-brain connectome.** Parcellate the brain into many regions and compute the correlation between *every pair*. The output is an N x N matrix and a network. Good when you want a holistic view of the brain's coupling structure rather than a single region's profile.

We will compute the **connectome** view in §2.2, then come back in §2.3 to discuss the seed-to-voxel extension conceptually.

> ❓ **Questions:**
>
> 1. The GLM and the FC analyses run on the **same BOLD** with the **same motion confounds**. Why might they still produce different pictures of the language network?
>
> 2. A connectome with 39 regions has 39 x 38 / 2 = 741 unique edges. With voxel-level FC the number of unique pairs is in the billions. Beyond computational cost, what statistical problem does that create?


### 2.2 🐍 Whole-brain group connectome with an atlas

To draw the connectome we parcellate the brain into regions, extract one time series per region, and compute the correlation between every pair. Doing this voxel-by-voxel would be too noisy and too large to look at, so we use an atlas.

We use the [**MSDL atlas**](https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_atlas_msdl.html), 39 functionally-defined probabilistic regions with precomputed MNI coordinates. For each subject we extract one time series per region with [`NiftiMapsMasker`](https://nilearn.github.io/stable/modules/generated/nilearn.maskers.NiftiMapsMasker.html), compute the 39 x 39 correlation matrix, **Fisher-z transform** it, and average across subjects. The group-mean Fisher-z matrix (back-transformed to r for plotting) is the dataset's group FC connectome.


In [ ]:
%%bash
# Stage the MSDL atlas from the shared course folder, same pattern as
# every other dataset in this notebook.
mkdir -p results/sess-2
rm -rf  results/sess-2/msdl_atlas
cp  -r  /data/teaching/costantinoai/sess-2/msdl_atlas results/sess-2/
chmod -R u+w results/sess-2/msdl_atlas


In [ ]:
from nilearn.datasets import fetch_atlas_msdl
from nilearn.maskers import NiftiMapsMasker
from nilearn.connectome import ConnectivityMeasure
from nilearn.plotting import plot_matrix, plot_connectome

# Resolve the local copy of MSDL (already staged by the %%bash cell above).
# fetch_atlas_msdl just resolves paths inside data_dir without re-downloading.
# docs: https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_atlas_msdl.html
msdl = fetch_atlas_msdl(data_dir=str(RESULTS_DIR))
print(f'MSDL: {len(msdl.labels)} regions')

# ConnectivityMeasure encapsulates the per-subject correlation/partial-correlation/
# tangent-space recipe used in the connectivity literature, so we do not have to
# write the matrix maths by hand.
# docs: https://nilearn.github.io/stable/modules/generated/nilearn.connectome.ConnectivityMeasure.html
corr_measure = ConnectivityMeasure(kind='correlation', standardize='zscore_sample')

z_mats = []
for sub in subjects:
    bold = layout.get(subject=sub, task=task, scope='derivatives',
                      desc='preproc', extension='.nii.gz', return_type='filename')[0]

    cnf  = pd.read_csv(layout.get(subject=sub, task=task, scope='derivatives',
                       desc='confounds', extension='.tsv', return_type='filename')[0],
                       sep='\t').rename(columns=rename)[motion_cols]

    masker = NiftiMapsMasker(
        maps_img=msdl.maps,
        standardize='zscore_sample',
        standardize_confounds=True,
        low_pass=0.1, high_pass=0.01, t_r=t_r,
    )
    ts = masker.fit_transform(bold, confounds=cnf.values)
    r = corr_measure.fit_transform([ts])[0]
    np.fill_diagonal(r, 0)
    # Average correlation matrices across subjects in Fisher-z space. The
    # group mean of z is a better summary than the group mean of r.
    z_mats.append(np.arctanh(np.clip(r, -0.999, 0.999)))

# Average in Fisher-z space, then back-transform to r for plotting.
group_r_mat = np.tanh(np.mean(z_mats, axis=0))

# Reordered correlation matrix: plot_matrix(reorder=True) clusters the rows
# and columns so that tightly-coupled regions sit next to each other, which
# is what makes the canonical large-scale networks pop out as diagonal blocks.
fig = plt.figure(figsize=(8, 7))
plot_matrix(group_r_mat, labels=msdl.labels, vmin=-.8, vmax=.8,
            reorder=True, figure=fig,
            title=f'group MSDL FC (n={len(z_mats)}, mean Fisher-z back to r)')
plt.show()

# Same matrix, drawn as a brain network: each node sits at its MNI coordinate
# and edges show the strongest connections (top 10% by absolute correlation).
plot_connectome(group_r_mat, msdl.region_coords,
                edge_threshold='95%', node_size=20,
                title=f'top 5% group FC edges (n={len(z_mats)})')
plt.show()


The reordered group matrix shows tight blocks on the diagonal: the canonical large-scale networks (default-mode, dorsal attention, visual, motor). They emerge cleanly from the group average even on a 5-minute task scan, because subject-specific noise averages out in Fisher-z space and the shared connectivity structure survives. The connectome view shows the same information as long-range edges between MNI-located nodes. By preserving only 5% of the connections, we can highlight strong inter-hemispering coupling, and some posterior-to-anterior connections, in areas similar to the ones we observed in the GLM section above.

> 📖 **Explore further:**
>
> * [Nilearn seed-to-voxel example](https://nilearn.github.io/stable/auto_examples/03_connectivity/plot_seed_to_voxel_correlation.html), the canonical pipeline we use in §2.3.
> * [Nilearn connectivity gallery](https://nilearn.github.io/stable/auto_examples/03_connectivity/index.html), partial correlation, sparse inverse covariance, dynamic FC.
> * [`ConnectivityMeasure`](https://nilearn.github.io/stable/modules/generated/nilearn.connectome.ConnectivityMeasure.html), see `kind='partial correlation'` and `kind='tangent'` for alternative measures that go beyond pairwise Pearson.
> * [`load_confounds_strategy`](https://nilearn.github.io/stable/modules/generated/nilearn.interfaces.fmriprep.load_confounds_strategy.html), recommended preset confound models (Friston-24, aCompCor, scrubbing).
> * **Beta-series correlation** (Rissman et al. 2004), task-modulated FC by correlating per-trial GLM betas instead of raw time series.


### 2.3 Extension: seed-based FC

The connectome above gives every region equal billing, 39 nodes, 741 edges, one diagonal-block matrix. A complementary lens is **seed-based FC**, which zooms in on a single region and asks the rest of the brain to report in.

**The idea**:

1. Pick one **seed**, a single coordinate or small ROI. A natural choice for this dataset would be the peak of the §1.9 group GLM in left inferior frontal gyrus.
2. Extract the seed's time series, and the time series of *every voxel in the brain*.
3. Correlate the seed against each voxel, Fisher-z transform, and group-average exactly as we did for the connectome.
4. The result is a whole-brain map, each voxel's value is its coupling strength with the seed.

**What it shares with the connectome.** The substrate is identical, the same preprocessed BOLD, the same motion confounds, the same Fisher-z group recipe. Both are *unsupervised with respect to the task*, neither uses `events.tsv`. Both express coupling as Pearson r in the same low-frequency band.

**What changes.** The connectome treats all atlas regions symmetrically, you read it as a *graph* (nodes and edges). Seed FC privileges one region and produces a *map* with the same spatial resolution as the BOLD. So the two answer different questions:

| Question | Use |
|---|---|
| "What network structure does this brain have?" | connectome |
| "What does **this specific region** talk to?" | seed FC |

**A subtle gotcha for task data.** Both methods include task-driven co-fluctuations in their correlations, if two regions both ramp up during language blocks they will correlate even if they do not communicate directly. For task-free FC you would either run a resting-state scan, or regress the task design out of the BOLD before computing correlations. We have not done either here, so read the maps as *coupling during a language task*, not *intrinsic coupling*.


## 3. Representational Similarity Analysis (RSA)

### 3.1 The RSA idea

The GLM in [§1](#1.-GLM-with-pyBIDS-and-nilearn-(first-and-second-level)) tells you *where* in the brain a condition activates. RSA asks a different question: **how does a region organise its many conditions relative to each other?**

Two stimuli that produce *similar* multivariate response patterns are represented as close by that region; two stimuli with *dissimilar* patterns are far apart. Stack all pairwise dissimilarities into an $n \times n$ matrix and you get a **Representational Dissimilarity Matrix (RDM)**, a compact fingerprint of the region's representational geometry.

Because an RDM is just a matrix of distances between conditions, you can build one from anything: brain patterns, behavioural ratings, layer activations of a CNN, pixel arrays. And then you can **compare** them. RSA's strength is that it lets you put very different things into a common currency (Kriegeskorte, Mur & Bandettini 2008).

> 💡 **Intuition (two stimuli at a time).** Take two stimuli from the 92-image set, say `face_01` and `cat_01`. In a high-level visual region, both evoke similar voxel-pattern shapes: both faces of animals, broadly speaking. Their *distance* in the region's representational space is small. Now compare `face_01` and `bottle_01`: very different patterns, animate vs. inanimate, large distance. Repeat for all $\binom{92}{2}$ pairs and stack the distances into a matrix: that is the RDM. A region that organises stimuli by animacy has small distances *within* the animate block and *within* the inanimate block, and large distances *between* the two blocks. A region that organises stimuli by colour or shape will have completely different blocks. **The block structure of an RDM is the fingerprint of what the region cares about.**

> ❓ **Questions:**
>
> 1. What would the RDM look like for a region that does **not** distinguish any of the 92 stimuli (e.g. a chunk of CSF or a region that is not visually responsive)?
>
> 2. The RDM is symmetric and has zeros on the diagonal **by construction**. What would a non-zero diagonal mean if it ever appeared?
>
> 3. We use **dissimilarity** (1 - correlation) rather than similarity (correlation directly). Are these two ways of writing the same matrix, or do they encode different things?


### 3.2 🐍 The data

We will work with **two pre-staged datasets** copied from the shared course folder:

1. **Kriegeskorte 92-image set** (`92imageData/`): the original 92 colour photographs (humans, animals, plants, manmade objects), their per-stimulus binary attributes (`animate`, `face`, `body`, ...), and a few simulated/precomputed RDMs. Same source as the rsatoolbox demos.
2. **Algonauts 2019** (`algonauts2019/`): pre-computed fMRI **RDMs for human EVC and human IT** on the same 92 stimuli, from **15 subjects**, originally published by Cichy, Pantazis & Oliva (2014, *Nat. Neurosci.*; 2016, *Cereb. Cortex*) and re-distributed for the Algonauts Challenge. This gives us *real brain data* for two visual ROIs at opposite ends of the ventral stream.

Same copy-from-shared pattern as sess-1b.


In [ ]:
%%bash
# Copy both staged folders from /data/teaching/costantinoai/sess-2/ into your
# personal results/. We never write into the shared folder; results/ is yours.
mkdir -p results/sess-2

# 92imageData: Kriegeskorte stimuli + binary attributes + model/sim RDMs.
rm -rf results/sess-2/92imageData
cp -r /data/teaching/costantinoai/sess-2/92imageData results/sess-2/

# algonauts2019: real EVC + IT fMRI RDMs (15 subj) + the 92 jpg images.
rm -rf results/sess-2/algonauts2019
cp -r /data/teaching/costantinoai/sess-2/algonauts2019 results/sess-2/

# Make the local copies writable.
chmod -R u+w results/sess-2/92imageData results/sess-2/algonauts2019

# Quick listing so you can see what landed where.
ls -lh results/sess-2/92imageData/      | head
echo
ls -lh results/sess-2/algonauts2019/    | head


**What is in each folder.** `92imageData/` carries the original Kriegeskorte 2008 stimuli, their per-stimulus binary attributes (`animate`, `face`, `body`, ...), and a few precomputed/simulated RDMs (the simulated patterns we use in §3.4). `algonauts2019/` carries the Cichy et al. fMRI RDMs (one 92×92 matrix per subject per ROI) plus the 92 stimuli as JPG files, which we use both as **thumbnails** on the RDM axes and as **input to the pixel model** in §3.5.


In [ ]:
# Local paths to the two staged folders.
RSA_DIR  = RESULTS_DIR / '92imageData'
ALGO_DIR = RESULTS_DIR / 'algonauts2019'

# scipy reads classic .mat files (v <= 7.2). The Algonauts target_fmri.mat is
# an HDF5 v7.3 file, so we will use h5py for that one in §3.4.
from scipy.io import loadmat

# Load the Kriegeskorte 2008 supplemental: stimuli + binary category attributes.
# - stimuli_92objs: object array of dicts, one per stimulus (image + flags).
# - categoryVectors: 92 x 12 binary matrix (each row = stimulus, each col = attribute).
# - categoryLabels:  list of 12 attribute names matching the columns.
supp = loadmat(RSA_DIR / 'Kriegeskorte_Neuron2008_supplementalData.mat',
               simplify_cells=True)
stimuli         = supp['stimuli_92objs']        # array of 92 stimulus structs
category_labels = list(supp['categoryLabels'])  # 12 attribute names
category_mat    = supp['categoryVectors']       # (92, 12) binary

# Print the available attributes so you can see what the dataset annotates.
print('per-stimulus attributes:', category_labels)
print('category matrix shape:  ', category_mat.shape, '(92 stimuli x 12 attributes)')

# Show 12 example images so you can see what the dataset actually contains.
import matplotlib.pyplot as _plt
fig, axes = _plt.subplots(2, 6, figsize=(12, 5))
for ax, st in zip(axes.flat, stimuli[::8]):
    # Each stimulus carries its image as a numpy array under the 'image' field.
    ax.imshow(st['image']); ax.axis('off')
    # Tag each thumbnail with a category derived from the binary attributes.
    tag = 'animate' if (st['human'] or st['face'] or st['animal']) else 'inanimate'
    ax.set_title(tag, fontsize=9)
fig.suptitle('Kriegeskorte 2008: sample stimuli', y=1.02)
_plt.tight_layout(); _plt.show()


### 3.3 🐍 What is a multivariate pattern?

A **multivariate pattern** is just the response of every voxel in an ROI, lined up into a single row vector. In the GLM you ran in [§1.7](#1.7-🐍-Fitting-the-GLM-and-computing-a-contrast) you produced *two* such vectors per voxel grid (one $\beta$-map for `language`, one for `string`). RSA uses the same kind of object, just at scale: one $\beta$-map per condition, with an ROI mask applied so each $\beta$-map collapses into a single row vector. Stack all rows and you get a **patterns matrix** of shape `(n_conditions, n_voxels_in_ROI)`.

Concretely: in [sess-1a §3.5](sess-1a.ipynb) you saw that an ROI mask is just a NIfTI image with 1s inside the region and 0s outside; in [sess-1b §3.5](sess-1b.ipynb) you saw the preprocessed BOLD time series. The patterns matrix is what you get when you (a) fit a GLM to that BOLD, (b) compute one $\beta$-map per condition, and (c) apply a mask to keep only the voxels you care about. Step (c) is one line of nilearn:

```python
patterns = NiftiMasker(mask_img=roi_mask).fit_transform(beta_imgs)
# array of shape (n_conditions, n_voxels_in_ROI)
```

The clip below shows that step visually.


In [ ]:
Video('assets/sess-2/videos/ch01_patterns.mp4', width=640, embed=True)


### 3.4 🐍 Building the neural RDM with rsatoolbox

For each pair of conditions $(i, j)$ we want a single number that says how *different* the two response patterns are. The standard choice in RSA is the **correlation distance**:

$$d_{ij} = 1 - \mathrm{corr}(\mathbf{p}_i, \mathbf{p}_j)$$

Stack all $d_{ij}$ for the 92 stimuli and you get a symmetric $92 \times 92$ matrix with zeros on the diagonal, the **Representational Dissimilarity Matrix (RDM)**. 

We will use [`rsatoolbox`](https://rsatoolbox.readthedocs.io/) for everything RDM-related. The toolbox handles three things we want:

* `rsatoolbox.data.Dataset` packages a patterns matrix together with descriptors (which row is which condition, which group, which thumbnail to show, ...).
* `rsatoolbox.rdm.calc_rdm(...)` computes the RDM with a chosen distance.
* `rsatoolbox.vis.show_rdm(...)` plots it with the canonical Kriegeskorte-style colormap and (optionally) per-row stimulus thumbnails on the axes, which makes the structure of the RDM directly readable.

Before plotting anything we set up a few per-stimulus descriptors that every RDM in this section will reuse: a unique condition label, an animate / inanimate group label (used at plot time to **reorder rows** so the categorical structure jumps out), and a per-stimulus thumbnail icon (used as axis labels). We also wrap `show_rdm` in a small helper so each figure looks the same: every panel gets **its own colorbar** (no shared scale across ROIs), the inner white grid is suppressed, and the figure is large enough that the icons stay legible.


In [ ]:
import rsatoolbox
from rsatoolbox.vis.icon import Icon

# Per-stimulus descriptors, all in canonical 92-image order.
#   conds : a unique key per stimulus, required by calc_rdm so it does not
#           average rows that share a label.
#   group : 'animate' / 'inanimate' label, used at plot time to reorder rows
#           and to make the categorical structure visible in the RDM.
#   icons : a thumbnail used as axis "tick label" on each RDM figure.
animate = category_mat[:, category_labels.index('animate')].astype(int)
group   = np.where(animate == 1, 'animate', 'inanimate')
conds   = np.array([f'img_{i:02d}' for i in range(len(stimuli))])
icons   = np.array([Icon(image=s['image'], make_square=True) for s in stimuli])

# All RDMs in this section share the same per-stimulus descriptors. Wrap the
# one repetitive constructor call in a helper so the per-RDM cells below
# stay focused on what actually differs (the dissimilarity values).
def wrap_rdm(arr_2d, name, descriptor_key='roi', measure='1 - r'):
    """Turn a single 92x92 numpy array into an rsatoolbox.rdm.RDMs."""
    return rsatoolbox.rdm.RDMs(
        dissimilarities       = arr_2d[np.newaxis],
        dissimilarity_measure = measure,
        rdm_descriptors       = {descriptor_key: np.array([name])},
        pattern_descriptors   = {'conds': conds, 'icons': icons, 'group': group},
    )


def plot_rdm(rdms, title=None, figsize=(7, 7), **kwargs):
    """rsatoolbox.vis.show_rdm with our house style:

    * each panel gets its own colorbar (show_colorbar='panel'), so RDMs
      with different ranges are not forced onto a shared scale.
    * the default white inner grid (one line every num_pattern_groups cells,
      drawn by the rsatoolbox mplstyle) is suppressed -- it looks like a
      checkerboard on top of the RDM.
    * icons are stacked 2-deep on each axis (num_pattern_groups=2), which
      gives them a bit more room than the default 1-row layout.
    """
    args = dict(
        pattern_descriptor = 'icons',
        cmap               = 'classic',
        num_pattern_groups = 2,
        show_colorbar      = 'panel',
        figsize            = figsize,
        icon_spacing       = 1.0,
    )
    args.update(kwargs)
    fig, axarr, _ = rsatoolbox.vis.show_rdm(rdms, **args)
    for ax in np.atleast_1d(axarr).flat:
        if ax is not None:
            ax.grid(False)            # drop the white grid drawn by the mplstyle
    if title:
        fig.suptitle(title, y=1.02)
    plt.show()


With the helper in place we load the **real fMRI RDMs** from Cichy, Pantazis & Oliva (2014, 2016): 15 subjects watched the 92 stimuli in an fMRI scanner; the authors defined two ROIs per subject, **EVC** (early visual cortex, V1+V2 retinotopic) and **IT** (inferior temporal cortex, high-level visual), and computed one $92\times 92$ correlation-distance RDM per subject per ROI. We get **15 RDMs per ROI**, which we average across subjects for the figure and keep individually for the per-subject scoring step in §3.6.

Before plotting the RDMs, a quick anatomical reminder of *where* these ROIs sit in the brain.


**Where in the brain are these ROIs?** The Cichy et al. masks were defined per subject and are not bundled with the RDM file, but to give the figure context we can plot **approximate** EVC and IT regions from the **Harvard-Oxford** cortical atlas you used in [sess-1a §3.5](sess-1a.ipynb). The atlas is *illustrative only*, not the actual masks Cichy et al. used.


In [ ]:
from nilearn.datasets import fetch_atlas_harvard_oxford
from nilearn import image as nimg
from nilearn.plotting import plot_glass_brain

# Same atlas as sess-1a §2.5 (Harvard-Oxford lateralised cortical, 2 mm).
ho = fetch_atlas_harvard_oxford('cortl-maxprob-thr25-2mm', data_dir=str(RESULTS_DIR))
ho_img    = nimg.load_img(ho.maps)                   # 3D label image
ho_data   = ho_img.get_fdata()                       # (X, Y, Z) integer labels
ho_labels = list(ho.labels)                          # 0='Background', then region names
print('atlas shape:', ho_data.shape, '  n labels:', len(ho_labels))

# Pull the integer label for any region that contains the given substring.
def labels_for(substring):
    return [i for i, name in enumerate(ho_labels) if substring.lower() in name.lower()]

# EVC ~= occipital pole + lateral occipital cortex (early/posterior visual).
evc_indices = labels_for('Occipital Pole') + labels_for('Lateral Occipital Cortex')
# IT  ~= inferior temporal gyrus + temporal occipital fusiform (high-level visual).
it_indices  = (labels_for('Inferior Temporal Gyrus') +
               labels_for('Temporal Occipital Fusiform Cortex'))
print('EVC label indices:', evc_indices)
print('IT  label indices:', it_indices)

# Build a binary mask image per ROI by collecting voxels whose atlas label
# matches any of the indices we just selected.
def mask_for(indices):
    mask = np.isin(ho_data, indices).astype(np.int8)
    return nimg.new_img_like(ho_img, mask)

evc_mask = mask_for(evc_indices)
it_mask  = mask_for(it_indices)

# Two glass-brain panels side by side. Same plotting routine as sess-1a §2.5
# (plot_roi was used there); plot_glass_brain gives a similar bird's-eye view.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_glass_brain(evc_mask, axes=axes[0], display_mode='lyrz',
                 title='Approximate EVC (Harvard-Oxford)', colorbar=False)
plot_glass_brain(it_mask,  axes=axes[1], display_mode='lyrz',
                 title='Approximate IT (Harvard-Oxford)',  colorbar=False)
plt.show()


In [ ]:
import h5py

# Algonauts target_fmri.mat is HDF5 v7.3 (MATLAB's modern format).
# scipy.io.loadmat cannot read this version; h5py can.
algo_path = ALGO_DIR / 'target_fmri.mat'
with h5py.File(algo_path, 'r') as f:
    # h5py reads MATLAB axes in reverse order, so the README's (15, 92, 92)
    # appears here as (92, 92, 15). Transpose to (n_subjects, 92, 92) for clarity.
    it_subj  = np.transpose(np.array(f['IT_RDMs']),  (2, 0, 1))
    evc_subj = np.transpose(np.array(f['EVC_RDMs']), (2, 0, 1))

print('IT  RDMs:', it_subj.shape, '   EVC RDMs:', evc_subj.shape)

# Group-level RDM = mean across the 15 subjects, per ROI. Both arrays stay in
# canonical 92-image order; show_rdm reorders for the plot via subset_pattern.
it_mean  = it_subj.mean(axis=0)
evc_mean = evc_subj.mean(axis=0)


We plot **each ROI in its own figure**, with **its own colorbar**. This is deliberate: EVC and IT live on different dissimilarity ranges, and forcing one shared colorbar would flatten the structure of whichever ROI has the smaller range. With per-panel scaling, the within-ROI structure stays legible in both.


In [ ]:
# Each ROI -> its own RDMs object -> its own figure with its own colorbar.
# subset_pattern() reorders rows so animate stimuli sit on top, which makes
# the categorical block structure jump out in IT.
rdm_it  = wrap_rdm(it_mean,  'IT')
rdm_evc = wrap_rdm(evc_mean, 'EVC')

plot_rdm(rdm_it.subset_pattern('group',  ['animate', 'inanimate']),
         title='Human IT RDM  (group mean, n=15, Cichy 2014)',
         rdm_descriptor='roi')

plot_rdm(rdm_evc.subset_pattern('group', ['animate', 'inanimate']),
         title='Human EVC RDM  (group mean, n=15, Cichy 2014)',
         rdm_descriptor='roi')


Read the two RDMs side by side. In **IT**, you should see two darker (low-dissimilarity) blocks on the diagonal: animate-vs-animate top-left, inanimate-vs-inanimate bottom-right, with brighter (high-dissimilarity) off-diagonal blocks. That block structure is what people mean when they say IT carries an *animacy code*. In **EVC**, the same row reordering produces no such block: dissimilarities are dominated by low-level image statistics (contrast, spatial layout, colour) which do not respect the animate / inanimate boundary.

That visual impression is exactly the hypothesis we will test in §3.6 by comparing each ROI's RDM to two **model RDMs**, which we build next.


### 3.5 🐍 Two model RDMs: animacy and pixel-level

A **model RDM** is a 92 × 92 matrix that encodes a hypothesis: *if my idea about how the brain organises these stimuli were true, this is what the dissimilarity structure should look like.* By correlating a model RDM with each subject's neural RDM we ask which hypothesis the brain best matches.

We build two very different models, both straight from the dataset:

1. **Animacy** (semantic): dissimilarity is **0** for two stimuli of the same animacy class and **1** otherwise. A purely categorical model that lives only in stimulus-label space. This is how we "predict" the RDM would look like if an area encoded only Animacy.
2. **Pixel-level** (visual): take the raw stimulus images, flatten each to a long pixel vector, and compute pairwise correlation distance between those vectors. This captures only how the images *look*, not what they depict.

If a region's neural RDM correlates well with the **animate** model but not the **pixel** model, that region is doing something *semantic* that goes beyond image statistics. The opposite pattern is what we expect from early visual cortex.


In [ ]:
# ---------- Animacy model ----------
# Outer-not-equal on the binary `animate` vector: 1 if two stimuli are in
# different animacy classes, 0 if same. That is the whole hypothesis.
animate_rdm_arr = (animate[:, None] != animate[None, :]).astype(float)

rdm_animate = wrap_rdm(animate_rdm_arr, 'animacy model',
                       descriptor_key='name', measure='0/1 (categorical)')

plot_rdm(rdm_animate.subset_pattern('group', ['animate', 'inanimate']),
         title='Animacy model RDM  (0 = same class, 1 = different)',
         rdm_descriptor='name')


The **animacy** model captures *what the stimuli depict*. The **pixel** model below captures the complementary side, *how the stimuli look*: we treat each image as a long vector of RGB pixel values and compute the same correlation distance between every pair. No semantics involved at all.


In [ ]:
# ---------- Pixel-level model ----------
# Each row = one stimulus, flattened to a long RGB vector. No semantics, just
# raw pixel values. Same correlation distance, completely different feature space.
from PIL import Image

target_size = (175, 175)                          # original Kriegeskorte canvas
pixels_arr = np.stack([
    np.asarray(Image.open(ALGO_DIR / '92images' / f'image_{i:02d}.jpg')
               .convert('RGB').resize(target_size), dtype=float).reshape(-1)
    for i in range(1, 93)
])
print('pixel matrix:', pixels_arr.shape, '(92 stimuli x flattened pixels)')

ds_pixel  = rsatoolbox.data.Dataset(
    measurements    = pixels_arr,
    obs_descriptors = {'conds': conds},
)
rdm_pixel = rsatoolbox.rdm.calc_rdm(ds_pixel, method='correlation',
                                    descriptor='conds')

# Re-attach the descriptors calc_rdm did not forward, so plot_rdm can put
# icons on the axes and reorder by group.
rdm_pixel.pattern_descriptors['icons'] = icons
rdm_pixel.pattern_descriptors['group'] = group
rdm_pixel.rdm_descriptors['name']      = np.array(['pixel model'])

plot_rdm(rdm_pixel.subset_pattern('group', ['animate', 'inanimate']),
         title='Pixel-level model RDM  (1 - r between flattened images)',
         rdm_descriptor='name')


The two model RDMs look very different. **Animacy** is a hard checkerboard: two blocks of 0s on the diagonal and 1s everywhere else, because it only knows the binary class label. **Pixel** is continuous and patchy: pairs of stimuli with similar lighting, background or spatial layout end up close together regardless of what they depict, so the animate / inanimate split is mostly invisible.

That is exactly what we want, two hypotheses that disagree about as much as two RDMs can. If they made the same predictions, comparing them would tell us nothing.


### 3.6 🐍 Quantitative comparison: which model fits which ROI?

We now have everything we need: 15 per-subject RDMs for IT, 15 for EVC, and two model RDMs (animacy, pixel). For every (ROI, model) pair we compute a Spearman rank correlation between the model RDM and each subject's RDM, then summarise the 15 scores as mean ± SEM.


In [ ]:
# Pack the 15 per-subject RDMs of each ROI into a single rsatoolbox.RDMs,
# and stack the two model RDMs into one models object too. compare() then
# returns all (n_models x n_subjects) Spearman scores in one vectorised call.

def make_subj_rdms(arr3d, roi_name):
    """Wrap a (n_subjects, 92, 92) numpy array as an rsatoolbox.RDMs."""
    return rsatoolbox.rdm.RDMs(
        dissimilarities       = arr3d,
        dissimilarity_measure = '1 - r',
        rdm_descriptors       = {'roi':     np.array([roi_name] * arr3d.shape[0]),
                                 'subject': np.arange(arr3d.shape[0])},
        pattern_descriptors   = {'conds': conds},
    )

rdms_it_subj  = make_subj_rdms(it_subj,  'IT')
rdms_evc_subj = make_subj_rdms(evc_subj, 'EVC')

# make sure both model RDMs have the SAME dissimilarity_measure ---
rdm_animate.dissimilarity_measure = 'model'
rdm_pixel.dissimilarity_measure   = 'model'

# Stack the two model RDMs along the rdm-axis -> one RDMs with n_rdm=2.
models_rdms = rsatoolbox.rdm.concat([rdm_animate, rdm_pixel])

# compare(models, data, method='spearman') returns shape (n_models, n_data).
# Each row is one model's per-subject score across the 15 subjects.
scores_it  = rsatoolbox.rdm.compare(models_rdms, rdms_it_subj,  method='spearman')
scores_evc = rsatoolbox.rdm.compare(models_rdms, rdms_evc_subj, method='spearman')

# Bar chart: 2 ROIs (groups) x 2 models (bars per group), with SEM across subjects.
rois          = ['IT', 'EVC']
model_names   = ['animate', 'pixel']
roi_to_scores = {'IT': scores_it, 'EVC': scores_evc}
colours       = {'animate': '#4a90d9', 'pixel': '#d97a4a'}
x, width      = np.arange(len(rois)), 0.38

fig, ax = plt.subplots(figsize=(7, 4.5))
for m_idx, mname in enumerate(model_names):
    means = [roi_to_scores[roi][m_idx].mean() for roi in rois]
    sems  = [roi_to_scores[roi][m_idx].std(ddof=1) /
             np.sqrt(roi_to_scores[roi][m_idx].size) for roi in rois]
    ax.bar(x + (m_idx - 0.5) * width, means, width, yerr=sems,
           capsize=4, label=mname, color=colours[mname])

ax.axhline(0, color='black', lw=0.5)
ax.set_xticks(x); ax.set_xticklabels(rois)
ax.set_ylabel('Spearman rho  (model vs subject RDM)')
ax.set_title('Real data: which model best matches each ROI? (mean +/- SEM, n=15)')
ax.legend(title='model RDM')
plt.tight_layout(); plt.show()

for roi in rois:
    for m_idx, mname in enumerate(model_names):
        s = roi_to_scores[roi][m_idx]
        print(f'  {roi:3s}  {mname:8s}  rho = {s.mean():+.3f} +/- '
              f'{s.std(ddof=1)/np.sqrt(s.size):.3f}  (n={s.size})')


The pattern flips between regions: the **animate** model wins in **IT**, the **pixel** model wins in **EVC**. Both regions have representational structure on the same 92 stimuli; they just organise them by very different criteria, exactly as the ventral-stream literature predicts (Kriegeskorte 2008, Cichy et al. 2014).

That bar chart is RSA in one shot: real brain data, two ROIs, two transparent hypotheses, and a quantitative answer at the group level.

> **What is missing for a real publication.** A single $\rho$ per ROI per model says nothing about *significance*. A real RSA pipeline adds (i) a more careful distance like cross-validated Mahalanobis (`crossnobis`), (ii) a **noise-ceiling** estimate so you know how high any model could possibly score given the between-subject variability, and (iii) permutation-based statistics. See [`rsatoolbox.inference`](https://rsatoolbox.readthedocs.io/en/stable/inference.html) when you graduate from this notebook.

> ❓ **Questions:**
>
> 1. The Algonauts file ships **one RDM per subject per ROI**, not per-voxel patterns. Why is the per-subject RDM enough for our model comparison, but not enough if we wanted to do a *univariate* analysis or visualise the underlying patterns?
>
> 2. The IT vs EVC figure shows the **animate** model winning in IT and the **pixel** model winning in EVC. What kind of model do you think would win in the Parahippocampal Place Area (PPA)?

> 📖 **Explore further:**
> * [Kriegeskorte, Mur & Bandettini (2008)](https://www.frontiersin.org/articles/10.3389/neuro.06.004.2008/full), the original RSA paper.
> * [Cichy, Pantazis & Oliva (2014, 2016)](https://userpage.fu-berlin.de/rmcichy/fusion_project_page/main.html), the source of the 15-subject EVC and IT RDMs you just analysed.
> * [`rsatoolbox` documentation](https://rsatoolbox.readthedocs.io/en/stable/) and the [demos folder](https://github.com/rsagroup/rsatoolbox/tree/main/demos).
> * [Algonauts Project](http://algonauts.csail.mit.edu/), the challenge that re-released these RDMs in a clean `.mat` form.


### 3.7 🐍 Cross-species RSA: monkey IT vs human IT

So far the comparison has been *within humans*: human IT vs human EVC, both scored against two model RDMs. The same machinery works when one of the RDMs comes from a completely different measurement, in a completely different species. The 92-stimulus dataset ships with one of the most cited RDMs in the field: **monkey IT**, computed from single-unit recordings of **674 IT neurons in two macaques** (Kriegeskorte et al. 2008, *Neuron*). Same stimuli, same matrix shape, completely different physical signal (action potentials per neuron vs BOLD per voxel).

This is where RSA earns its keep as a **method**, not just a plotting trick. An RDM strips away the substrate (voxels, neurons, units of a CNN, behavioural ratings) and keeps only the **representational geometry**, the relative distances between conditions. Once two systems are in this common currency, you can ask:

* Does **monkey IT** look like **human IT**? (homology across species)
* Does it look like **human EVC**? (probably not, if IT is doing something different from V1/V2)
* How does monkey IT score against the same **animacy** and **pixel** models?
* And, by extension, this is exactly the same question you ask of a CNN layer or a behavioural similarity-judgement RDM.

That is the teaching moment: a single 92×92 matrix lets you put electrophysiology, fMRI, behaviour and deep nets on the same axis.


In [ ]:
# 92_modelRDMs.mat ships several pre-computed RDMs. We pull the monkey IT one
# (Kriegeskorte et al. 2008, Neuron: 674 IT neurons, 2 macaques, same 92 stimuli).
models_mat        = loadmat(RSA_DIR / '92_modelRDMs.mat', simplify_cells=True)
monkey_it_rdm_arr = next(m['RDM'] for m in models_mat['Models']
                                 if m['name'] == 'monkeyIT')
print('monkey IT RDM:', monkey_it_rdm_arr.shape, monkey_it_rdm_arr.dtype)

rdm_monkey_it = wrap_rdm(monkey_it_rdm_arr, 'monkey IT (674 neurons)')
plot_rdm(rdm_monkey_it.subset_pattern('group', ['animate', 'inanimate']),
         title='Monkey IT RDM  (Kriegeskorte 2008, 674 single units)',
         rdm_descriptor='roi')


The monkey IT RDM should look **strikingly similar in structure** to the human IT RDM you plotted in §3.4: the same animate / inanimate block on the diagonal, the same darker face cluster inside the animate block. That visual similarity is the famous Kriegeskorte 2008 result: high-level visual cortex is organised by similar categorical axes in macaques and humans, despite ~25 million years of evolutionary divergence and a completely different recording technique.

To turn that impression into a number, we score the monkey RDM against (i) every human subject's IT and EVC RDM and (ii) the two model RDMs. A single Spearman $\rho$ per comparison is enough to make the point.


In [ ]:
# Compare the monkey IT RDM to four targets:
# - human IT subjects: cross-species similarity at same processing stage
# - human EVC subjects: similarity to earlier visual cortex
# - animacy model: category-level structure
# - pixel model: low-level image statistics

rho = {
    'human IT  (n=15)': rsatoolbox.rdm.compare(
        rdm_monkey_it, rdms_it_subj,  method='spearman').ravel(),
    'human EVC (n=15)': rsatoolbox.rdm.compare(
        rdm_monkey_it, rdms_evc_subj, method='spearman').ravel(),
    'animacy model':    rsatoolbox.rdm.compare(
        rdm_monkey_it, rdm_animate,   method='spearman').ravel(),
    'pixel model':      rsatoolbox.rdm.compare(
        rdm_monkey_it, rdm_pixel,     method='spearman').ravel(),
}

# Print summary stats (mean ± SEM for humans, single value for models)
for name, vals in rho.items():
    if vals.size > 1:
        print(f'  monkey IT vs {name:18s} rho = {vals.mean():+.3f} '
              f'+/- {vals.std(ddof=1)/np.sqrt(vals.size):.3f}')
    else:
        print(f'  monkey IT vs {name:18s} rho = {vals[0]:+.3f}')


# Split into brain vs brain and brain vs model
human_labels = ['human IT  (n=15)', 'human EVC (n=15)']
model_labels = ['animacy model', 'pixel model']

human_means = [rho[k].mean() for k in human_labels]
human_sems  = [rho[k].std(ddof=1)/np.sqrt(rho[k].size) for k in human_labels]

model_means = [rho[k].mean() for k in model_labels]
model_sems  = [0.0 for _ in model_labels]

human_colours = ['#4a90d9', '#7bb3e8']
model_colours = ['#4adb89', '#d97a4a']


# Use independent y-axes (sharey=False) because the scales can differ:
# human correlations often cluster tightly, while model fits can differ more.
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharey=False)

axes[0].bar(human_labels, human_means, yerr=human_sems,
            capsize=4, color=human_colours)
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_title('Monkey IT vs human cortex')
axes[0].set_ylabel('Spearman rho')

axes[1].bar(model_labels, model_means, yerr=model_sems,
            capsize=4, color=model_colours)
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('Monkey IT vs model RDMs')

fig.suptitle('Cross-species RSA: what explains monkey IT representational structure?')

plt.tight_layout()
plt.show()

Read the bars left to right. **Monkey IT** lines up best with **human IT**, less well with **human EVC**, and shows the same model fingerprint as human IT did in §3.6: high correlation with the **animacy** model, much lower with the **pixel** model. Two species, two recording techniques (extracellular spikes vs BOLD), one shared representational geometry.

This is the conceptual punchline of RSA. Anything that produces a 92×92 dissimilarity matrix can be slotted into the same comparison: a CNN layer (run the 92 stimuli through ResNet, take any layer's features, compute pairwise correlation distance), behavioural similarity ratings (16 subjects judged how similar the 92 images look, see `92_behavRDMs.mat` in this folder), or a wholly different brain area in a wholly different species. Put them all on the same axis and the question stops being *what is the signal* and becomes *what is the geometry*. That is why a method invented for fMRI now spans systems neuroscience, cognitive science, and computational modelling.

> 💡 **Where to take this next.** The Kriegeskorte 2008 paper compared monkey IT to human IT and found a very similar geometry, which is the result the bar chart above replicates. Yamins et al. (2014) and Khaligh-Razavi & Kriegeskorte (2014) extended the same logic to deep networks: each CNN layer becomes one RDM, and you can ask which layer best matches monkey/human IT. The `92_behavRDMs.mat` file in this folder contains 16 subjects' similarity-judgement RDMs on the same stimuli, ready to drop into the same `compare` call if you want to add a behavioural row to the comparison.


---

## Wrap-up: three lenses on the same brain

You ran three analyses on real fMRI data. Each asked a different question of the same kind of object (a $\beta$ map or a BOLD time series), and each used a tool that handled most of the heavy lifting for you:

| Lens | Question | Output | Key library call |
|---|---|---|---|
| **GLM** (§1) | *Where* does the brain respond more to A than to B? | a thresholded contrast map | `nilearn.glm.first_level.FirstLevelModel` + `SecondLevelModel` |
| **FC** (§2)  | *Which regions co-fluctuate* with a given seed (or with each other)? | a correlation map / a connectome matrix | `nilearn.maskers.*` + `nilearn.connectome.ConnectivityMeasure` |
| **RSA** (§3) | How does a region *organise* its many conditions relative to each other? | a 92×92 dissimilarity matrix you can compare to anything | `rsatoolbox.rdm.calc_rdm` + `compare` |

The aha is that GLM and FC ran on the *same* language-localizer data and sit on the same scan (§2), and that RSA could in principle take the very $\beta$-maps you produced in §1.7 and turn them into a representational geometry, the same way it turned monkey IT spike rates into one (§3.7). One dataset, three lenses; or one method, many substrates.

The exercises below ask you to *break* one analysis choice at a time (drop the motion regressors, swap correlation for Euclidean, change the threshold) and check whether the conclusion still stands. That is the practical version of "is this result robust?".


## 📝 Exercises

Now it is your turn. Using what you have learned about the GLM, RSA, and FC, complete the exercises below. The point of these is not to copy a result but to see how stable each method is when you change one analysis choice at a time.

---

**Exercise 1: Beta maps vs the contrast (GLM)**

a. Re-fit the smoothed GLM and compute the **language** beta map and the **string** beta map separately, with `output_type='effect_size'`.

b. Plot all three maps (`language`, `string`, `language - string`) side by side at the same threshold (e.g. `threshold=3.1` for the z-map; pick a sensible cutoff for the betas).

c. Which regions are present in both beta maps, and which ones are contrast-specific? Write 2-3 sentences explaining what the contrast is doing that the individual betas are not.
> 💡 hint: re-use the `smoothed_model` from §1.7 and call `compute_contrast(name, output_type='effect_size')` for `'language'` and `'string'`.


In [ ]:
# Exercise 1: beta maps vs contrast



---

**Exercise 2: What do the nuisance regressors do? (GLM)**

a. Re-build the design matrix **without** the 6 motion regressors (drop `add_regs` and `add_reg_names`).

b. Re-fit the smoothed GLM and compute the same `language - string` z-map.

c. Compare it to the original z-map. Did the language clusters become cleaner, noisier, or roughly unchanged? Why?


In [ ]:
# Exercise 2: GLM without motion regressors



---

**Exercise 3: Try a different dissimilarity (RSA)**

a. In the neural RDM cell, change `method='correlation'` to `method='euclidean'` and re-run.

b. Compare the new RDM matrix to the original one. Does the block structure stay visible?

c. Re-run the model comparison with the new RDM. Does the **animate** model still beat the **pixel** model?

d. Explain in one sentence why correlation distance is often preferred when the overall amplitude of a beta pattern is not the effect of interest.
> 💡 hint: change `method='correlation'` to `method='euclidean'` in the `calc_rdm` call in §3.4. The score cell in §3.6 needs no change.


In [ ]:
# Exercise 3: RSA with Euclidean distance



---

**Exercise 4: Redefine the category (RSA)**

a. Modify the `animate` definition so that **only `human`** images count as animate (not `face` or `animal`).

b. Rebuild the categorical model RDM and compare it to the neural RDM.

c. Does this stricter definition match the brain better or worse than the original? What does that tell you about how the simulated IT cortex carves up the stimuli?
> 💡 hint: in §3.5 the line that builds `animate_bin = category_mat[:, animate_idx]` is what you change. Use only `category_mat[:, category_labels.index('human')]`.


In [ ]:
# Exercise 4: RSA with a stricter animacy definition



---

**Exercise 5: Threshold the connectome (FC)**

a. Re-run the glass-brain plot with `edge_threshold='95%'` (top 5% of edges only).

b. Re-run it with `edge_threshold='50%'` (top 50% of edges).

c. Re-run with 'partial correlation' as measure.

d. Which threshold gives the most readable picture, and which gives the most complete picture? Where would you draw the line if you had to put one figure in a paper?
> 💡 hint: in §2.2 the only argument that needs changing is `edge_threshold` passed to `plot_connectome`.


In [ ]:
# Exercise 5: FC edge thresholds

